In [5]:
# =============================================================================
# OEB Dataset Statistics (Simple Version)
# =============================================================================
from pathlib import Path
import json
import re
import pandas as pd
import numpy as np

# --- CONFIG ---
COLLECTION = "OEB"
DATA_DIR = Path("/work/data/processed")
SHORT_FILE = DATA_DIR / f"{COLLECTION}_resumen.json"
LONG_FILE  = DATA_DIR / f"{COLLECTION}_texto.json"

In [6]:
# --- Load JSON ---
def load_docs(path):
    with open(path, "r", encoding="utf-8") as f:
        obj = json.load(f)
    docs = obj if isinstance(obj, list) else obj.get("documents", obj.get("data", []))
    rows = []
    for d in docs:
        md = d.get("metadata", {}) if isinstance(d.get("metadata"), dict) else {}
        rows.append({
            "item_key": md.get("item_key") or d.get("item_key"),
            "parent_key": md.get("parent_key") or d.get("parent_key"),
            "text": d.get("text", ""),
        })
    df = pd.DataFrame(rows)
    # Filter out template rows and empty text
    df = df.dropna(subset=["text", "item_key"])
    df = df[df["text"].str.strip() != ""]
    df = df[~df["item_key"].astype(str).str.endswith(("#", "$"), na=False)]
    return df.reset_index(drop=True)

short_df = load_docs(SHORT_FILE)
long_df = load_docs(LONG_FILE)

print(f"Loaded: {len(short_df)} short, {len(long_df)} long")

Loaded: 47513 short, 47513 long


In [10]:
# --- Compute Statistics ---

# Word counts (simple split)
short_df["word_count"] = short_df["text"].str.split().str.len()
long_df["word_count"] = long_df["text"].str.split().str.len()

# Has numbers? (simple regex check)
short_df["has_numbers"] = short_df["text"].str.contains(r"\d", regex=True)

# Basic counts
total_items = short_df["item_key"].nunique()
base_templates = short_df["parent_key"].nunique()
expansion_factor = total_items / base_templates

# Short-format stats
short_mean = short_df["word_count"].mean()
short_median = short_df["word_count"].median()
short_std = short_df["word_count"].std()

# Long-format stats
long_mean = long_df["word_count"].mean()
long_median = long_df["word_count"].median()
long_std = long_df["word_count"].std()

# Numeric queries
with_nums = short_df["has_numbers"].sum()
without_nums = len(short_df) - with_nums

# Near-duplicate clusters (>1 variants per template)
parent_counts = short_df.groupby("parent_key").size()
clusters_gt = (parent_counts > 1).sum()
items_in_clusters = parent_counts[parent_counts > 1].sum()

In [11]:
# --- Print LaTeX Table ---
print(f"""\\begin{{table}}[ht]
\\centering
\\caption{{OEB Subcategory Dataset Statistics}}
\\label{{tab:dataset_statistics}}
\\begin{{tabular}}{{|l|r|}}
\\hline
\\textbf{{Characteristic}} & \\textbf{{Value}} \\\\
\\hline
Total unique items & {total_items:,} \\\\
Base templates & {base_templates:,} \\\\
Average expansion factor & {expansion_factor:.1f} \\\\
\\hline
\\textit{{Short-Format description ('resumen')}} (query) & \\\\
\\quad Mean length (words) & {short_mean:.1f} \\\\
\\quad Median length (words) & {int(short_median)} \\\\
\\quad Std deviation (words) & {short_std:.1f} \\\\
\\hline
\\textit{{Long-Format description ('texto')}} (target) & \\\\
\\quad Mean length (words) & {long_mean:.1f} \\\\
\\quad Median length (words) & {int(long_median)} \\\\
\\quad Std deviation (words) & {long_std:.1f} \\\\
\\hline
Queries with numeric parameters & {with_nums:,} ({100*with_nums//len(short_df)}\\%) \\\\
Queries without numeric parameters & {without_nums:,} ({100*without_nums//len(short_df)}\\%) \\\\
Single-item templates ($=$1 variants) & {clusters_gt:,} ({100*clusters_gt//total_items}\\%) \\\\
\\hline
\\end{{tabular}}
\\end{{table}}""")

\begin{table}[ht]
\centering
\caption{OEB Subcategory Dataset Statistics}
\label{tab:dataset_statistics}
\begin{tabular}{|l|r|}
\hline
\textbf{Characteristic} & \textbf{Value} \\
\hline
Total unique items & 47,513 \\
Base templates & 30 \\
Average expansion factor & 1583.8 \\
\hline
\textit{Short-Format description ('resumen')} (query) & \\
\quad Mean length (words) & 19.7 \\
\quad Median length (words) & 20 \\
\quad Std deviation (words) & 2.4 \\
\hline
\textit{Long-Format description ('texto')} (target) & \\
\quad Mean length (words) & 84.5 \\
\quad Median length (words) & 83 \\
\quad Std deviation (words) & 15.0 \\
\hline
Queries with numeric parameters & 47,430 (99\%) \\
Queries without numeric parameters & 83 (0\%) \\
Near-duplicate clusters ($>$1 variants) & 25 (0\%) \\
\hline
\end{tabular}
\end{table}
